# **Name: Manahil Shahid**
# **Reg No: SP24-BDS-023**
# **Class: BDS-6A**
# **Subject: Natural Language Processing**
# **Lab 02: Lab Tasks**

## Setup

In [1]:
# Install once if needed: pip install pdfplumber python-docx reportlab nltk pandas requests

import re
import os
import csv
import json
import math
import sys
import requests
import pdfplumber
import pandas as pd
import nltk
from collections import Counter
from docx import Document
from docx.oxml.ns import qn
from docx.text.paragraph import Paragraph
from nltk.corpus import stopwords

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Creating my sample files

The tasks need a PDF file and some Word files to work on. To keep the notebook complete and easy to run anywhere, I am creating my own small sample files with code.

The PDF has six pages about natural language processing. The first five pages have text, and the sixth page is left blank on purpose, so I can show how my code handles a page with zero words. It also has a web address and a few numbers in the text, which I need for the cleaning task.

I am also creating two Word files. Document A has headings, normal paragraphs, an empty paragraph, a bullet paragraph and one table. Document B has some headings that are the same as Document A, some that are different, and two paragraphs that are repeated from Document A, so I can test the comparison and duplicate removal task.

If I want to use my own files instead, I only need to change the file names in the tasks below.

In [2]:
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
from reportlab.lib.utils import simpleSplit

# ---------- sample PDF with 6 pages (the last page is blank) ----------
pdf_pages = [
    "Natural language processing is a field of artificial intelligence that helps computers understand human language. "
    "It combines ideas from linguistics, statistics, and computer science. In 2024 many companies started using language "
    "models in their daily products. You can read more about the field at https://www.example.com/nlp for a short "
    "introduction. The main goal of language processing is to turn raw text into something a machine can work with.",

    "Text preprocessing is the first step in most language projects. The text is cleaned so that the model does not get "
    "confused by useless symbols. Common steps include removing punctuation, converting text to lowercase, and splitting "
    "the text into words. Another useful step is removing stopwords, which are very common words such as the and is. "
    "After cleaning, the text is easier to count and compare. Good preprocessing can improve the results of a model by a "
    "large margin.",

    "Tokenization means breaking a piece of text into smaller units called tokens. A token is usually a word, but it can "
    "also be a sentence or a single character. Once the text is tokenized, we can count how often each word appears. "
    "Word frequency is a simple but powerful idea, because the most frequent words often reveal the main topic of a "
    "document. For example, a news article about cricket will repeat words like match, team, and score many times.",

    "Language processing is used in many everyday tools. Search engines use it to understand what a person is looking for. "
    "Email services use it to filter spam messages. Chat assistants use it to answer questions in natural language. "
    "Translation tools use it to convert text from one language to another. Companies also use it to study customer "
    "reviews and find out what people like or dislike about a product. These examples show how useful the field has become.",

    "Data quality matters a lot. A model can only learn from the text it is given. If the text is messy, the model will "
    "learn messy patterns. Clean and balanced data gives the best results.",

    ""   # blank page on purpose
]

pdf_path = "sample_lab.pdf"
c = canvas.Canvas(pdf_path, pagesize=A4)
for page_text in pdf_pages:
    y = 800
    for line in simpleSplit(page_text, "Helvetica", 12, 480):
        c.drawString(50, y, line)
        y -= 18
    c.showPage()
c.save()

# ---------- sample Word document A (has a table) ----------
doc_a = Document()
doc_a.add_paragraph("Lab Sample Document A", style="Title")
doc_a.add_paragraph("Introduction to Natural Language Processing", style="Heading 1")
doc_a.add_paragraph("Natural language processing helps computers read and understand human language.")
doc_a.add_paragraph("It is used in search engines, chat assistants, and translation tools every day.")
doc_a.add_paragraph("Short line here.")
doc_a.add_paragraph("Text Preprocessing", style="Heading 2")
doc_a.add_paragraph("Cleaning text means removing symbols, digits, and extra spaces before analysis.")
table = doc_a.add_table(rows=4, cols=3)
table.style = "Table Grid"
table_rows = [
    ["Step", "Tool", "Purpose"],
    ["Tokenize", "NLTK", "Split text into words"],
    ["Clean", "re", ""],
    ["Count", "Counter", "Find word frequency"],
]
for r, row in enumerate(table_rows):
    for col, value in enumerate(row):
        table.cell(r, col).text = value
doc_a.add_paragraph("")
doc_a.add_paragraph("Applications", style="Heading 1")
doc_a.add_paragraph("Applications include spam filtering, sentiment analysis, and machine translation for many languages.", style="Body Text")
doc_a.add_paragraph("Sentiment analysis studies whether a review sounds positive or negative.")
doc_a.add_paragraph("Spam filtering", style="List Bullet")
doc_a.save("sample_doc_a.docx")

# ---------- sample Word document B ----------
doc_b = Document()
doc_b.add_paragraph("Lab Sample Document B", style="Title")
doc_b.add_paragraph("Introduction to Natural Language Processing", style="Heading 1")
doc_b.add_paragraph("  Natural Language Processing helps computers read and understand human language.  ")
doc_b.add_paragraph("Language models learn patterns from very large collections of text.")
doc_b.add_paragraph("Text Preprocessing", style="Heading 2")
doc_b.add_paragraph("Cleaning text means removing symbols, digits, and extra spaces before analysis.")
doc_b.add_paragraph("Tokenization splits text into smaller pieces such as words and sentences.")
doc_b.add_paragraph("Evaluation of Models", style="Heading 1")
doc_b.add_paragraph("Accuracy and precision are two common ways to measure how well a model works.")
doc_b.add_paragraph("Future Work", style="Heading 1")
doc_b.add_paragraph("Future work will focus on larger datasets and better multilingual support.")
doc_b.save("sample_doc_b.docx")

print("Created:", pdf_path, "sample_doc_a.docx", "sample_doc_b.docx")

Created: sample_lab.pdf sample_doc_a.docx sample_doc_b.docx


### What the output shows

The last line confirms that the three files were created in the same folder as my notebook. From now on every task reads these files.

## Task 1: Multi Page PDF Extraction with Word Frequency Analysis

In this task I am reading the PDF page by page, turning the text into a clean list of words, counting the words, and then searching for a keyword. I am solving it in four parts.

### Part a: Extract text from each page

Here I am opening the PDF with pdfplumber and looping through its pages one by one. For each page I call extract_text() and store the result in a list, so index 0 holds page 1, index 1 holds page 2, and so on. A page with no text gives back None, so I am using `or ""` to turn that into an empty string. Finally I print how many pages I extracted.

In [3]:
pages_text = []

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        text = page.extract_text() or ""
        pages_text.append(text)

print("Total number of pages extracted:", len(pages_text))

Total number of pages extracted: 6


#### What the output shows

The output says 6 pages were extracted. My sample PDF has five pages with text and one blank page at the end, and the blank page still counts as a page in the list. It just holds an empty string, so the list stays in the same order as the PDF.

### Part b: Merge pages and build a token list

Here I am joining all page strings into one long string with a space between them. Then I use one re.sub() call to delete everything that is not a letter or a space, which removes both punctuation and digits together. After that I change everything to lowercase and use split() to cut the text on whitespace into a list of words, which are called tokens. At the end I print the total token count.

In [4]:
merged_text = " ".join(pages_text)

clean_text = re.sub(r'[^a-zA-Z\s]', '', merged_text)

tokens = clean_text.lower().split()

print("Total token count:", len(tokens))
print("First 15 tokens:", tokens[:15])

Total token count: 338
First 15 tokens: ['natural', 'language', 'processing', 'is', 'a', 'field', 'of', 'artificial', 'intelligence', 'that', 'helps', 'computers', 'understand', 'human', 'language']


#### What the output shows

The token count is 338, which means the whole PDF turned into 338 lowercase words after punctuation and digits were removed. The first 15 tokens show the result clearly, because the text starts with natural language processing and everything is lowercase with no commas or full stops attached to the words.

One small thing I noticed is that the web address on page one has no spaces inside it, so after removing the punctuation it became one long token. The year 2024 disappeared completely, because it was made only of digits.

### Part c: Word frequency after removing stopwords

Here I am writing my own list of common stopwords (more than the 20 that the task asks for). Then I loop through the tokens and, for every token that is not a stopword, I add one to its count in a dictionary. At the end I sort the dictionary by count from highest to lowest and print the top 10 words.

In [5]:
my_stopwords = {
    "the", "a", "an", "is", "are", "was", "were", "of", "to", "in", "and", "or", "it", "that",
    "this", "for", "on", "as", "with", "by", "can", "be", "from", "such", "also", "at", "so",
    "not", "than", "how", "what", "we", "you", "its", "their", "which", "into", "about", "but",
    "does", "will", "has", "have", "each", "one", "many", "most", "only", "more", "very"
}
print("Number of stopwords I am removing:", len(my_stopwords))

word_freq = {}
for word in tokens:
    if word not in my_stopwords:
        word_freq[word] = word_freq.get(word, 0) + 1

top_10 = sorted(word_freq.items(), key=lambda item: item[1], reverse=True)[:10]

print()
print("Top 10 most frequent words:")
for word, count in top_10:
    print(f"{word}: {count}")

Number of stopwords I am removing: 50

Top 10 most frequent words:
text: 11
language: 8
use: 5
model: 4
words: 4
processing: 3
field: 3
word: 3
natural: 2
understand: 2


#### What the output shows

I removed 50 stopwords, which is more than the 20 needed. The top 10 list shows that text is the most frequent meaningful word with 11 appearances, followed by language with 8. Words like use, model, words and processing come next.

This makes sense because the whole PDF is about text and language processing, so the most frequent words tell me the main topic of the document. This is exactly why removing stopwords helps, because without it the top of the list would be filled with words like the and is.

### Part d: Search for a keyword and show its sentences

Here I am writing a small function that takes a keyword and the page list. For each page I first replace line breaks with spaces, then split the text into sentences wherever a full stop, question mark or exclamation mark is followed by a space. I check every sentence in lowercase, and if it contains the keyword I print the page number and the sentence. If nothing is found after checking all pages, I print a message saying so.

The keyword itself comes from the user using input(). After that I call the function a second time with a word that is not in the PDF, to show the no match message.

In [6]:
def search_keyword(keyword, pages):
    keyword = keyword.lower()
    found = False
    for page_number, page_text in enumerate(pages, start=1):
        flat_text = page_text.replace("\n", " ")
        sentences = re.split(r'(?<=[.!?])\s+', flat_text)
        for sentence in sentences:
            if keyword in sentence.lower():
                print(f"Page {page_number}: {sentence.strip()}")
                found = True
    if not found:
        print(f'No sentence contains the keyword "{keyword}".')

keyword = input("Enter a keyword to search: ")
search_keyword(keyword, pages_text)

Enter a keyword to search: language
Page 1: Natural language processing is a field of artificial intelligence that helps computers understand human language.
Page 1: In 2024 many companies started using language models in their daily products.
Page 1: The main goal of language processing is to turn raw text into something a machine can work with.
Page 2: Text preprocessing is the first step in most language projects.
Page 4: Language processing is used in many everyday tools.
Page 4: Chat assistants use it to answer questions in natural language.
Page 4: Translation tools use it to convert text from one language to another.


#### What the output shows

The keyword I typed was language, and the output printed 7 sentences that contain it. Each line starts with its page number, so I can see that the word appears on page 1, page 2 and page 4. Pages 3 and 5 have no sentence with this word, so they are not shown.

The search checks each sentence in lowercase, so it finds the word whether it is written with a capital letter or not. For example, Language at the start of a sentence on page 4 was found too.

In [7]:
# Testing the no match case with a word that is not in the PDF
search_keyword("quantum", pages_text)

No sentence contains the keyword "quantum".


#### What the output shows

This time I searched for the word quantum, which is not written anywhere in the PDF. My code printed the no match message instead of printing nothing, so the user always gets a clear answer.

## Task 2: Structured Page Metadata Extraction and CSV Export

In this task I am collecting information about every page of the PDF in a clean structure, finding the longest and shortest pages, searching for a phrase, and saving a summary into a CSV file. I am solving it in four parts.

### Part a: Build a dictionary for each page

Here I am opening the PDF again and creating one dictionary per page with exactly four keys. page_number is the page position starting from 1. raw_text is the text of the page. word_count is the number of words, which I get by splitting the text on whitespace. char_count is the number of characters without any spaces, which I get by removing all whitespace with re.sub() and then measuring the length. All dictionaries go into a list called pages_data, and I print the first two of them.

In [8]:
pages_data = []

with pdfplumber.open(pdf_path) as pdf:
    for page_number, page in enumerate(pdf.pages, start=1):
        text = page.extract_text() or ""
        pages_data.append({
            "page_number": page_number,
            "raw_text": text,
            "word_count": len(text.split()),
            "char_count": len(re.sub(r'\s', '', text))
        })

print("Number of pages in the PDF:", len(pages_data), "(minimum needed is 5)")
print()
print("First 2 entries of pages_data:")
for entry in pages_data[:2]:
    print(json.dumps(entry, indent=2))

Number of pages in the PDF: 6 (minimum needed is 5)

First 2 entries of pages_data:
{
  "page_number": 1,
  "raw_text": "Natural language processing is a field of artificial intelligence that helps computers\nunderstand human language. It combines ideas from linguistics, statistics, and computer\nscience. In 2024 many companies started using language models in their daily products.\nYou can read more about the field at https://www.example.com/nlp for a short introduction.\nThe main goal of language processing is to turn raw text into something a machine can\nwork with.",
  "word_count": 67,
  "char_count": 383
}
{
  "page_number": 2,
  "raw_text": "Text preprocessing is the first step in most language projects. The text is cleaned so that\nthe model does not get confused by useless symbols. Common steps include removing\npunctuation, converting text to lowercase, and splitting the text into words. Another useful\nstep is removing stopwords, which are very common words such as the and i

#### What the output shows

The PDF has 6 pages, which is above the minimum of 5. The two dictionaries printed here belong to page 1 and page 2, and each has exactly the four keys page_number, raw_text, word_count and char_count.

Page 1 has 67 words and 383 characters without spaces, and page 2 has 79 words and 394 characters without spaces. In raw_text the symbol \n shows where a new line starts in the PDF, because the text was written on several lines on the page.

### Part b: Highest, lowest and average word count

Here I am using pages_data to answer three questions. For the highest word count I use max() with the word_count value as the key, and I print the whole content of that page. For the lowest non zero word count I first make a new list that only keeps pages with at least one word, and then use min() on it, so the blank page does not spoil the answer. For the average I add up all word counts and divide by the number of pages, rounded to two decimal places. I also print which pages had zero words, so those pages are clearly handled.

In [9]:
highest = max(pages_data, key=lambda page: page["word_count"])
print("Page with the highest word count:", highest["page_number"], "with", highest["word_count"], "words")
print("Full content of that page:")
print(highest["raw_text"])
print()

non_zero_pages = [page for page in pages_data if page["word_count"] > 0]
zero_pages = [page["page_number"] for page in pages_data if page["word_count"] == 0]

if non_zero_pages:
    lowest = min(non_zero_pages, key=lambda page: page["word_count"])
    print("Page with the lowest non zero word count:", lowest["page_number"], "with", lowest["word_count"], "words")
else:
    print("Every page is empty, so there is no lowest non zero word count.")

if pages_data:
    average_words = round(sum(page["word_count"] for page in pages_data) / len(pages_data), 2)
    print("Average word count across all pages:", average_words)

print("Pages with zero words:", zero_pages if zero_pages else "None")

Page with the highest word count: 3 with 80 words
Full content of that page:
Tokenization means breaking a piece of text into smaller units called tokens. A token is
usually a word, but it can also be a sentence or a single character. Once the text is
tokenized, we can count how often each word appears. Word frequency is a simple but
powerful idea, because the most frequent words often reveal the main topic of a
document. For example, a news article about cricket will repeat words like match, team,
and score many times.

Page with the lowest non zero word count: 5 with 35 words
Average word count across all pages: 56.5
Pages with zero words: [6]


#### What the output shows

Page 3 has the highest word count with 80 words, and its full text is printed below the result. Page 5 has the lowest non zero word count with 35 words, because it is a short page. The average word count across all 6 pages is 56.5.

Page 6 is the blank page, and the output lists it as a page with zero words. Because I removed it before looking for the lowest count, it did not wrongly become the answer. It still counts in the average, which is why the average is a little lower than the average of only the pages with text.

### Part c: Search a phrase with a custom exception

Here I am first creating my own error type called PhraseNotFoundError. It is a class that inherits from Exception, and it lets me raise an error with a clear name when a phrase is missing.

Then I write a function that searches every page for the phrase. Before counting, I turn each page into one line of lowercase text, so capital letters and line breaks do not matter. The count() method tells me how many times the phrase appears on that page. If no page contains the phrase, the function raises PhraseNotFoundError. Outside the function I use try and except to catch that error and print a friendly message instead of letting the program crash.

I am running it once with input() and once more with a phrase that does not exist.

In [10]:
class PhraseNotFoundError(Exception):
    pass

def search_phrase(phrase, data):
    phrase_lower = phrase.lower()
    matches = []
    for page in data:
        one_line_text = " ".join(page["raw_text"].split()).lower()
        count = one_line_text.count(phrase_lower)
        if count > 0:
            matches.append((page["page_number"], count))
    if not matches:
        raise PhraseNotFoundError(f'The phrase "{phrase}" was not found on any page.')
    for page_number, count in matches:
        print(f"Page {page_number}: the phrase appears {count} time(s)")

phrase = input("Enter a phrase to search: ")
try:
    search_phrase(phrase, pages_data)
except PhraseNotFoundError as error:
    print("Search result:", error)

Enter a phrase to search: natural language
Page 1: the phrase appears 1 time(s)
Page 4: the phrase appears 1 time(s)


#### What the output shows

I typed the phrase natural language with small letters. The output shows that it appears once on page 1 and once on page 4. In the PDF those pages write it as Natural language at the start and as natural language inside a sentence, so the search works even when the capital letters are different.

In [11]:
# Testing the error case with a phrase that is not in the PDF
try:
    search_phrase("deep sea diving", pages_data)
except PhraseNotFoundError as error:
    print("Search result:", error)

Search result: The phrase "deep sea diving" was not found on any page.


#### What the output shows

The phrase deep sea diving is not in the PDF, so the function raised PhraseNotFoundError. My try and except block caught it and printed a friendly message. The program did not crash, which is exactly what the task asks for.

### Part d: Write and re read a CSV file

Here I am using Python's built in csv module, not pandas. I open a file called pdf_summary.csv for writing and use DictWriter to write a header row with the three columns page_number, word_count and char_count. Then I write one row for every page. To confirm that the file was saved correctly, I open it again for reading, use csv.reader, and print every row one by one.

In [12]:
columns = ["page_number", "word_count", "char_count"]

with open("pdf_summary.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=columns)
    writer.writeheader()
    for page in pages_data:
        writer.writerow({
            "page_number": page["page_number"],
            "word_count": page["word_count"],
            "char_count": page["char_count"]
        })

print("Contents of pdf_summary.csv:")
with open("pdf_summary.csv", "r", encoding="utf-8") as file:
    reader = csv.reader(file)
    for row in reader:
        print(row)

Contents of pdf_summary.csv:
['page_number', 'word_count', 'char_count']
['1', '67', '383']
['2', '79', '394']
['3', '80', '369']
['4', '78', '381']
['5', '35', '150']
['6', '0', '0']


#### What the output shows

The printed rows are the same numbers I calculated earlier, which proves the file was written and read back correctly. The first row is the header with the three column names, and after it there is one row for each page, from page 1 to page 6.

Page 6 shows 0 for both the word count and the character count, because it is the blank page. The csv module reads every value back as text, which is why the numbers appear inside quotation marks.

## Task 3: PDF Text Preprocessing Pipeline for NLP

In this task I am building a full cleaning pipeline. I start with raw PDF text and finish with a file of clean sentences that is ready for NLP work. I am solving it in four parts.

### Part a: Extract and clean the text

Here I am extracting all the text of the PDF into one single string. I replace the line breaks with spaces first, because otherwise the last word of one line would stick to the first word of the next line once the line breaks are deleted.

Then I apply four cleaning steps in the exact order the task gives, each one with re.sub(). Step one removes web addresses. Step two removes digits. Step three removes every character that is not a letter, a space or a full stop. Step four turns many spaces in a row into a single space. I print the character count before and after cleaning, and a short preview of the cleaned text.

In [13]:
with pdfplumber.open(pdf_path) as pdf:
    all_text = " ".join((page.extract_text() or "").replace("\n", " ") for page in pdf.pages)

print("Characters before cleaning:", len(all_text))

cleaned = re.sub(r'https?://\S+|www\.\S+', '', all_text)   # step 1: remove URLs
cleaned = re.sub(r'\d', '', cleaned)                        # step 2: remove digits
cleaned = re.sub(r'[^A-Za-z. ]', '', cleaned)               # step 3: keep only letters, spaces and full stops
cleaned = re.sub(r' +', ' ', cleaned)                       # step 4: collapse extra spaces

print("Characters after cleaning:", len(cleaned))
print()
print("Preview of cleaned text:")
print(cleaned[:250])

Characters before cleaning: 2016
Characters after cleaning: 1970

Preview of cleaned text:
Natural language processing is a field of artificial intelligence that helps computers understand human language. It combines ideas from linguistics statistics and computer science. In many companies started using language models in their daily produ


#### What the output shows

The text had 2016 characters before cleaning and 1970 characters after cleaning, so the cleaning removed 46 characters. These were mostly the web address, the digits of the year 2024, and the commas.

The preview shows the result nicely. The full stops are still there because the task asks to keep them, but the commas are gone. For example, the words linguistics statistics and computer science are no longer separated by commas. The year 2024 is also gone, so the words In and many now sit next to each other.

### Part b: Lowercase and split into sentences

Here I am changing the cleaned text to lowercase and splitting it into sentences using re.split() on the characters full stop, question mark and exclamation mark. The split leaves extra spaces around each sentence and can also leave empty strings, for example after the very last full stop. So I strip each sentence and then remove the empty ones. I print the sentence count before and after removing the empty strings.

In [14]:
cleaned = cleaned.lower()

sentences = re.split(r'[.!?]', cleaned)
print("Sentence count before removing empty strings:", len(sentences))

sentences = [sentence.strip() for sentence in sentences]
sentences = [sentence for sentence in sentences if sentence != ""]
print("Sentence count after removing empty strings:", len(sentences))

Sentence count before removing empty strings: 28
Sentence count after removing empty strings: 27


#### What the output shows

Splitting the text gave 28 pieces, and after removing the empty strings I have 27 real sentences. The extra empty piece comes from the very last full stop, because there is nothing written after it, and the split still creates an empty string there.

Removing empty strings is important, because an empty sentence would be counted as a sentence and would cause problems in the later steps.

### Part c: Remove stopwords and drop short sentences

Here I am loading the English stopword list from nltk. For each sentence I split it into words, keep only the words that are not stopwords, and check how many words are left. If a sentence has fewer than 5 words left, I throw it away. The sentences I keep are stored as the remaining words joined back together with spaces, so they are ready for NLP use. I print the sentence count before and after the filter and show 3 sample sentences.

In [15]:
english_stopwords = set(stopwords.words('english'))

print("Sentence count before the filter:", len(sentences))

filtered_sentences = []
for sentence in sentences:
    words = sentence.split()
    remaining_words = [word for word in words if word not in english_stopwords]
    if len(remaining_words) >= 5:
        filtered_sentences.append(" ".join(remaining_words))

print("Sentence count after the filter:", len(filtered_sentences))
print()
print("3 sample sentences:")
for sample in filtered_sentences[:3]:
    print("*", sample)

Sentence count before the filter: 27
Sentence count after the filter: 24

3 sample sentences:
* natural language processing field artificial intelligence helps computers understand human language
* combines ideas linguistics statistics computer science
* many companies started using language models daily products


#### What the output shows

Before the filter I had 27 sentences, and after the filter I have 24. So 3 sentences were dropped for having fewer than 5 meaningful words left. These were short sentences like Data quality matters a lot, where only four words remain after the stopwords are removed.

The three samples show what the sentences look like now. Small words like is, of, that and the are gone, and only the important words remain, for example natural language processing field artificial intelligence helps computers understand human language.

### Part d: Save to a file and check it

Here I am writing every remaining sentence into a file called nlp_ready_output.txt with one sentence on each line. Then I open the file again, read all lines, and print the total number of lines. I also use max() and min() with len as the key to find the longest and the shortest sentence by number of characters.

In [16]:
with open("nlp_ready_output.txt", "w", encoding="utf-8") as file:
    for sentence in filtered_sentences:
        file.write(sentence + "\n")

with open("nlp_ready_output.txt", "r", encoding="utf-8") as file:
    lines = [line.strip() for line in file.readlines()]

print("Total lines written:", len(lines))
print()

longest_sentence = max(lines, key=len)
shortest_sentence = min(lines, key=len)

print("Longest sentence (", len(longest_sentence), "characters ):")
print(longest_sentence)
print()
print("Shortest sentence (", len(shortest_sentence), "characters ):")
print(shortest_sentence)

Total lines written: 24

Longest sentence ( 99 characters ):
natural language processing field artificial intelligence helps computers understand human language

Shortest sentence ( 33 characters ):
examples show useful field become


#### What the output shows

The file has 24 lines, which matches the 24 sentences left after the filter, so nothing was lost while writing and reading. The longest sentence has 99 characters and it is the description of natural language processing from page 1. The shortest sentence has 33 characters, and it is examples show useful field become.

Every line is now clean lowercase text without stopwords, punctuation or numbers, which is the kind of text an NLP model prefers to work with.

## Task 4: DOCX Paragraph Classification and Statistical Analysis

In this task I am opening a Word file, looking at the style of every paragraph, sorting the paragraphs into groups, and calculating some numbers about the body text. I am using my sample Document A. I am solving it in four parts.

### Part a: Read paragraphs and their styles

Here I am opening the Word file with the python docx library. For every paragraph I store its text and its style name in a dictionary, and all dictionaries go into one list. A style name is the label Word gives a paragraph, like Normal, Title or Heading 1. I print the total number of paragraphs and the list of unique style names, which I get by putting the names into a set.

In [17]:
doc = Document("sample_doc_a.docx")

paragraph_data = []
for paragraph in doc.paragraphs:
    paragraph_data.append({"text": paragraph.text, "style": paragraph.style.name})

print("Total paragraph count:", len(paragraph_data))
print("Unique style names:", sorted(set(item["style"] for item in paragraph_data)))

Total paragraph count: 12
Unique style names: ['Body Text', 'Heading 1', 'Heading 2', 'List Bullet', 'Normal', 'Title']


#### What the output shows

The document has 12 paragraphs in total. The blank paragraph after the table is counted too, because Word treats an empty line as a paragraph. The unique style names are Body Text, Heading 1, Heading 2, List Bullet, Normal and Title.

Only six style names exist even though there are 12 paragraphs, since many paragraphs share the same style. The table cells are not counted here, because python docx keeps table text separate from the normal paragraphs.

### Part b: Classify paragraphs into three groups

Here I am sorting all paragraphs into exactly three groups. I check blank paragraphs first, because the task says blank ones belong to the Other group. After that, a paragraph goes to Headings if its style name starts with Heading. It goes to Body if the style is Normal or the style name contains the word Body. Everything else goes to Other or Empty. I print how many paragraphs are in each group.

In [18]:
headings = []
body = []
other_or_empty = []

for item in paragraph_data:
    style = item["style"]
    text = item["text"].strip()
    if text == "":
        other_or_empty.append(item)
    elif style.startswith("Heading"):
        headings.append(item)
    elif style == "Normal" or "Body" in style:
        body.append(item)
    else:
        other_or_empty.append(item)

print("Headings:", len(headings))
print("Body:", len(body))
print("Other or Empty:", len(other_or_empty))
print("Total:", len(headings) + len(body) + len(other_or_empty))

Headings: 3
Body: 6
Other or Empty: 3
Total: 12


#### What the output shows

There are 3 headings, 6 body paragraphs and 3 in the Other or Empty group. Together they make 12, which matches the total paragraph count, so no paragraph was missed or counted twice.

The 3 headings are two Heading 1 paragraphs and one Heading 2 paragraph. The 6 body paragraphs are five Normal paragraphs and one Body Text paragraph. The Other or Empty group has the Title, the List Bullet paragraph and the blank paragraph.

### Part c: Statistics of the body paragraphs

Here I am only looking at body paragraphs that have more than 5 words, because very short lines are not real sentences. For each of them I calculate the word count and the character count without spaces. Then I print the paragraph with the highest word count, the paragraph with the lowest word count, and the mean word count rounded to two decimal places.

In [19]:
qualifying = []
for item in body:
    words = item["text"].split()
    if len(words) > 5:
        qualifying.append({
            "text": item["text"],
            "word_count": len(words),
            "char_count": len(item["text"].replace(" ", ""))
        })

print("Body paragraphs with more than 5 words:", len(qualifying))
print()

if qualifying:
    highest = max(qualifying, key=lambda p: p["word_count"])
    lowest = min(qualifying, key=lambda p: p["word_count"])
    mean_words = round(sum(p["word_count"] for p in qualifying) / len(qualifying), 2)

    print("Highest word count:", highest["word_count"], "words,", highest["char_count"], "characters without spaces")
    print(highest["text"])
    print()
    print("Lowest word count above 5 words:", lowest["word_count"], "words,", lowest["char_count"], "characters without spaces")
    print(lowest["text"])
    print()
    print("Mean word count:", mean_words)
else:
    print("There are no qualifying body paragraphs.")

Body paragraphs with more than 5 words: 5

Highest word count: 13 words, 67 characters without spaces
It is used in search engines, chat assistants, and translation tools every day.

Lowest word count above 5 words: 10 words, 70 characters without spaces
Natural language processing helps computers read and understand human language.

Mean word count: 11.2


#### What the output shows

Five body paragraphs have more than 5 words. The short line Short line here was left out, because it has only 3 words. The paragraph with the highest word count is the one that starts with It is used in search engines, which has 13 words. The lowest one has 10 words, and the first paragraph with this count is the one about natural language processing helping computers.

The mean word count is 11.2. It is the total words of the five qualifying paragraphs divided by five.

### Part d: Nested dictionary of styles

Here I am building a dictionary where each style name points to another dictionary with a count and a list of texts. I loop through all paragraphs, add one to the count of its style, and add its text to the list. Then I sort the outer dictionary by count from the highest to the lowest. If a style has more than 2 paragraphs, I keep only the first 2 texts in its list, but the count still shows the real total. I print the result in a neat, indented form using json.dumps.

In [20]:
style_groups = {}
for item in paragraph_data:
    group = style_groups.setdefault(item["style"], {"count": 0, "texts": []})
    group["count"] += 1
    group["texts"].append(item["text"])

sorted_styles = dict(sorted(style_groups.items(), key=lambda pair: pair[1]["count"], reverse=True))

for style_name, info in sorted_styles.items():
    if info["count"] > 2:
        info["texts"] = info["texts"][:2]

print(json.dumps(sorted_styles, indent=2))

{
  "Normal": {
    "count": 6,
    "texts": [
      "Natural language processing helps computers read and understand human language.",
      "It is used in search engines, chat assistants, and translation tools every day."
    ]
  },
  "Heading 1": {
    "count": 2,
    "texts": [
      "Introduction to Natural Language Processing",
      "Applications"
    ]
  },
  "Title": {
    "count": 1,
    "texts": [
      "Lab Sample Document A"
    ]
  },
  "Heading 2": {
    "count": 1,
    "texts": [
      "Text Preprocessing"
    ]
  },
  "Body Text": {
    "count": 1,
    "texts": [
      "Applications include spam filtering, sentiment analysis, and machine translation for many languages."
    ]
  },
  "List Bullet": {
    "count": 1,
    "texts": [
      "Spam filtering"
    ]
  }
}


#### What the output shows

The result lists every style with its real count, and the styles are sorted from the highest count to the lowest. Normal is first with a count of 6. Its list of texts shows only the first 2 paragraphs, because it has more than 2. The other styles have 2 paragraphs or fewer, so all their texts are shown.

Heading 1 comes second with 2 paragraphs, and the styles with only 1 paragraph come after it. This one nested view gives me a quick summary of how the whole document is built.

## Task 5: DOCX Table Extraction and Pandas Integration

In this task I am pulling a table out of a Word file, turning it into a pandas DataFrame, finding the paragraphs that come before the table, and saving everything into a text file. I am using my sample Document A, which has one table. I am solving it in four parts.

### Part a: Extract every table

Here I am opening the Word file and looping through doc.tables. For each table I build a list of lists, where every inner list is one row and holds the text of each cell. I print the number of tables found and the size of each one written as rows by columns.

In [21]:
doc = Document("sample_doc_a.docx")

all_tables = []
for table in doc.tables:
    table_data = []
    for row in table.rows:
        table_data.append([cell.text for cell in row.cells])
    all_tables.append(table_data)

print("Total number of tables found:", len(all_tables))
for number, table_data in enumerate(all_tables, start=1):
    rows = len(table_data)
    columns = len(table_data[0]) if rows > 0 else 0
    print(f"Table {number}: {rows} rows x {columns} columns")

Total number of tables found: 1
Table 1: 4 rows x 3 columns


#### What the output shows

The document contains 1 table and its size is 4 rows by 3 columns. This includes the header row, so it has one header row and three data rows.

### Part b: Make a DataFrame and count empty cells

Here I am taking the first table, using its first row as the column names and the remaining rows as the data, and creating a pandas DataFrame. I print the full DataFrame. After that I count the empty cells in every column. A cell counts as empty if it is None or if it only has blank text.

In [22]:
first_table = all_tables[0]

df = pd.DataFrame(first_table[1:], columns=first_table[0])

print("Full DataFrame:")
print(df)
print()

empty_counts = df.apply(lambda column: sum(1 for value in column if value is None or str(value).strip() == ""))
print("Number of empty or None cells per column:")
print(empty_counts)

Full DataFrame:
       Step     Tool                Purpose
0  Tokenize     NLTK  Split text into words
1     Clean       re                       
2     Count  Counter    Find word frequency

Number of empty or None cells per column:
Step       0
Tool       0
Purpose    1
dtype: int64


#### What the output shows

The DataFrame uses the first row of the table, which is Step, Tool and Purpose, as column names. The other three rows became the data, so the DataFrame has 3 rows.

The empty cell count shows 0 for Step, 0 for Tool and 1 for Purpose. This is right, because in my table the Purpose cell in the Clean row was left empty on purpose.

### Part c: Paragraphs before the first table

Here I am going through the children of the document body in the order they appear in the file. The body contains paragraphs and tables mixed together, so this order tells me what comes first. As soon as I meet a table element, which is called w:tbl, I stop. Every paragraph element before that point is turned into a paragraph object and saved in a list. I print how many paragraphs were saved and also show their text.

In [23]:
pre_table_paragraphs = []

for child in doc.element.body.iterchildren():
    if child.tag == qn('w:tbl'):
        break
    if child.tag == qn('w:p'):
        pre_table_paragraphs.append(Paragraph(child, doc))

print("Number of paragraphs before the first table:", len(pre_table_paragraphs))
print()
for number, paragraph in enumerate(pre_table_paragraphs, start=1):
    print(f"{number}. [{paragraph.style.name}] {paragraph.text}")

Number of paragraphs before the first table: 7

1. [Title] Lab Sample Document A
2. [Heading 1] Introduction to Natural Language Processing
3. [Normal] Natural language processing helps computers read and understand human language.
4. [Normal] It is used in search engines, chat assistants, and translation tools every day.
5. [Normal] Short line here.
6. [Heading 2] Text Preprocessing
7. [Normal] Cleaning text means removing symbols, digits, and extra spaces before analysis.


#### What the output shows

There are 7 paragraphs before the first table. They are the title, the first heading, three normal paragraphs, the second heading and one more normal paragraph. The loop stopped at the table, so the paragraphs about applications that come after the table are not included.

Iterating through the body children in order is what makes this work, because doc.paragraphs alone would not tell me where the table sits between the paragraphs.

### Part d: Save both to one text file

Here I am writing one file called docx_extracted.txt with two labeled sections. The first section starts with the pre table paragraphs header line and then lists the paragraphs, one per line. The second section starts with the table data header line and then lists each table row with the cells joined by a space, a pipe sign and a space. After writing, I read the file back and print it to check it.

In [24]:
with open("docx_extracted.txt", "w", encoding="utf-8") as file:
    file.write("--- PRE-TABLE PARAGRAPHS ---\n")
    for paragraph in pre_table_paragraphs:
        file.write(paragraph.text + "\n")

    file.write("\n--- TABLE DATA ---\n")
    for row in first_table:
        file.write(" | ".join(row) + "\n")

with open("docx_extracted.txt", "r", encoding="utf-8") as file:
    print(file.read())

--- PRE-TABLE PARAGRAPHS ---
Lab Sample Document A
Introduction to Natural Language Processing
Natural language processing helps computers read and understand human language.
It is used in search engines, chat assistants, and translation tools every day.
Short line here.
Text Preprocessing
Cleaning text means removing symbols, digits, and extra spaces before analysis.

--- TABLE DATA ---
Step | Tool | Purpose
Tokenize | NLTK | Split text into words
Clean | re | 
Count | Counter | Find word frequency



#### What the output shows

The printed file has two clear sections. The first section starts with the pre table paragraphs header line and lists the 7 paragraphs. The second section starts with the table data header line and shows each row with the cells separated by pipe signs, such as Step | Tool | Purpose.

The row for Clean shows an empty space after the last pipe sign, which is the empty Purpose cell. This confirms that the file was written correctly.

## Task 6: Multi Document Comparison, Deduplication, and Frequency Report

In this task I am comparing two Word files, removing repeated paragraphs, counting words across both files, and printing a final summary report. Document A is sample_doc_a.docx and Document B is sample_doc_b.docx. I am solving it in four parts.

### Part a: Compare headings using sets

Here I am opening both files and collecting every paragraph whose style name starts with Heading into two lists. I strip extra spaces from each heading text. Then I turn the lists into sets, because sets make comparison easy. The & sign gives headings found in both documents, and the minus operator gives headings found in only one of them.

In [25]:
doc_a = Document("sample_doc_a.docx")
doc_b = Document("sample_doc_b.docx")

headings_a = [p.text.strip() for p in doc_a.paragraphs if p.style.name.startswith("Heading")]
headings_b = [p.text.strip() for p in doc_b.paragraphs if p.style.name.startswith("Heading")]

set_a = set(headings_a)
set_b = set(headings_b)

print("Headings common to both documents:")
for heading in sorted(set_a & set_b):
    print("*", heading)

print()
print("Headings only in Document A:")
for heading in sorted(set_a - set_b):
    print("*", heading)

print()
print("Headings only in Document B:")
for heading in sorted(set_b - set_a):
    print("*", heading)

Headings common to both documents:
* Introduction to Natural Language Processing
* Text Preprocessing

Headings only in Document A:
* Applications

Headings only in Document B:
* Evaluation of Models
* Future Work


#### What the output shows

Two headings are common to both documents, which are Introduction to Natural Language Processing and Text Preprocessing. The heading Applications is only in Document A. The headings Evaluation of Models and Future Work are only in Document B.

The set operations make this very simple. The & sign keeps what both sets share, and subtracting one set from the other keeps what only the first set has.

### Part b: Merge Normal paragraphs and remove duplicates

Here I am taking every paragraph with the style Normal from both files and joining them into one list. I skip blank paragraphs because they carry no text. To remove exact duplicates, I compare the stripped and lowercased version of each paragraph. I keep a set of texts I have already seen, and I only keep a paragraph the first time its comparison text appears. I print the count before and after.

In [26]:
normal_a = [p.text for p in doc_a.paragraphs if p.style.name == "Normal" and p.text.strip() != ""]
normal_b = [p.text for p in doc_b.paragraphs if p.style.name == "Normal" and p.text.strip() != ""]

merged_paragraphs = normal_a + normal_b
print("Total Normal paragraphs before deduplication:", len(merged_paragraphs))

seen = set()
unique_paragraphs = []
for paragraph in merged_paragraphs:
    key = paragraph.strip().lower()
    if key not in seen:
        seen.add(key)
        unique_paragraphs.append(paragraph.strip())

print("Total Normal paragraphs after deduplication:", len(unique_paragraphs))

Total Normal paragraphs before deduplication: 11
Total Normal paragraphs after deduplication: 9


#### What the output shows

Before deduplication there were 11 Normal paragraphs, and after deduplication there are 9. So 2 paragraphs were removed as duplicates.

One duplicate is the sentence about natural language processing helping computers. In Document B it has extra spaces and capital letters, but after stripping and lowercasing it matches the sentence in Document A. The other duplicate is the sentence about cleaning text, which is written exactly the same in both files.

### Part c: Combined word frequency

Here I am going through every unique paragraph, changing it to lowercase, and using re.sub() to remove punctuation. Then I split it into words and add all the words to one big list. I use collections.Counter to count how often each word appears. After that I remove my own list of stopwords, which has more than 15 words, and print the 20 most common words that are left.

In [27]:
all_words = []
for paragraph in unique_paragraphs:
    cleaned_paragraph = re.sub(r'[^\w\s]', '', paragraph.lower())
    all_words.extend(cleaned_paragraph.split())

word_counter = Counter(all_words)

stopwords_6 = {
    "the", "a", "an", "is", "are", "and", "or", "of", "to", "in", "it", "for", "on", "as",
    "with", "by", "from", "how", "well", "two", "very", "will", "such", "that", "this", "be"
}

for stop_word in stopwords_6:
    word_counter.pop(stop_word, None)

print("Stopwords removed:", len(stopwords_6))
print()
print("Top 20 most frequent words:")
for word, count in word_counter.most_common(20):
    print(f"{word}: {count}")

Stopwords removed: 26

Top 20 most frequent words:
language: 3
text: 3
analysis: 2
natural: 1
processing: 1
helps: 1
computers: 1
read: 1
understand: 1
human: 1
used: 1
search: 1
engines: 1
chat: 1
assistants: 1
translation: 1
tools: 1
every: 1
day: 1
short: 1


#### What the output shows

I removed 26 stopwords, which is more than the 15 needed. The top of the list is language and text with 3 each, then analysis with 2. All the other words appear only once.

Many words share the same count because my two documents are very small, so there are not many repeated words. On bigger documents this list would show much clearer differences between the words.

### Part d: Final summary

Here I am putting five numbers together in one summary. The total headings is the length of both heading lists added together. The unique body paragraphs is the length of my deduplicated list. The vocabulary size is the number of different words, which is the size of the set of all words. The average paragraph length is the total word count divided by the number of paragraphs, rounded to two decimal places. The longest paragraph is found by using max() with the word count as the key.

In [28]:
total_headings = len(headings_a) + len(headings_b)
total_unique_paragraphs = len(unique_paragraphs)
vocabulary_size = len(set(all_words))

paragraph_lengths = [len(paragraph.split()) for paragraph in unique_paragraphs]
average_length = round(sum(paragraph_lengths) / len(paragraph_lengths), 2)

longest_paragraph = max(unique_paragraphs, key=lambda paragraph: len(paragraph.split()))

print("Total headings across both documents:", total_headings)
print("Total unique body paragraphs:", total_unique_paragraphs)
print("Vocabulary size (unique word types):", vocabulary_size)
print("Average paragraph length in words:", average_length)
print()
print("Longest paragraph (", len(longest_paragraph.split()), "words ):")
print(longest_paragraph)

Total headings across both documents: 7
Total unique body paragraphs: 9
Vocabulary size (unique word types): 82
Average paragraph length in words: 10.33

Longest paragraph ( 14 words ):
Accuracy and precision are two common ways to measure how well a model works.


#### What the output shows

The summary shows 7 headings in total across both documents. Three come from Document A and four come from Document B, and the two common headings are counted once in each document. There are 9 unique body paragraphs and the vocabulary has 82 different words. The average paragraph length is 10.33 words.

The longest paragraph has 14 words and it is the sentence about accuracy and precision from Document B. This report gives a full picture of both documents in just a few lines.

## Task 7: Multi Endpoint API Data Joining and Analysis

From this task onwards I am working with live data from JSONPlaceholder, which is a free website that gives fake but realistic data for practice. An endpoint is just a web link that returns data. The data comes back in JSON format, which Python can turn into lists and dictionaries. These tasks need an internet connection to run. I am solving Task 7 in four parts.

### Part a: Fetch both endpoints safely

Here I am writing a small function called fetch_json. It sends a GET request to a link and checks the status code. A status code of 200 means the request worked. If the code is anything else, the function raises a ConnectionError and includes the real status code in the message. If the code is 200, it returns the data as a Python list. I use this function to fetch the users and the posts, and I print how many records I received from each.

In [ ]:
def fetch_json(url):
    response = requests.get(url, timeout=15)
    if response.status_code != 200:
        raise ConnectionError(f"Request to {url} failed with status code {response.status_code}")
    return response.json()

users = fetch_json("https://jsonplaceholder.typicode.com/users")
posts = fetch_json("https://jsonplaceholder.typicode.com/posts")

print("Users fetched:", len(users))
print("Posts fetched:", len(posts))

#### What the output shows

When I run this cell with a working internet connection, it prints the number of users and the number of posts. JSONPlaceholder normally gives 10 users and 100 posts. If the website ever answered with a status code other than 200, the function would stop with a ConnectionError that shows that exact code, so a bad response can never be used by mistake.

### Part b: Join users and posts

Here I am joining the two lists using the user id. First I make a dictionary that maps each user id to the user's name, so I can look a name up quickly. Then I create the result dictionary with one entry per user name, holding the email, a post count that starts at 0, and an empty list of titles. After that I go through every post, find the name of its author using userId, add one to the post count, and add the title to the list. I print the result for the first 3 users only.

In [ ]:
id_to_name = {user["id"]: user["name"] for user in users}

joined = {}
for user in users:
    joined[user["name"]] = {"email": user["email"], "post_count": 0, "titles": []}

for post in posts:
    author_name = id_to_name[post["userId"]]
    joined[author_name]["post_count"] += 1
    joined[author_name]["titles"].append(post["title"])

for name in list(joined.keys())[:3]:
    print(name)
    print(json.dumps(joined[name], indent=2))
    print()

#### What the output shows

The output shows three blocks, one for each of the first three users. Each block has the user's name, then the email, the post count and the list of all post titles written by that user. In the usual JSONPlaceholder data every user has written 10 posts, so each post count is 10 and each list holds 10 titles.

The join works because a post carries a userId and a user carries an id, and these two numbers are the link between the two lists.

### Part c: The user with the most posts

Here I am using max() on the joined dictionary, with post_count as the key, to find the user who wrote the most posts. The company name is not stored in the joined dictionary, so I look that user up in the original users list. Then I print the full name, email, company name and all post titles with numbers in front of them.

In [ ]:
top_name = max(joined, key=lambda name: joined[name]["post_count"])
top_info = joined[top_name]

top_user = next(user for user in users if user["name"] == top_name)

print("Full name:", top_name)
print("Email:", top_info["email"])
print("Company name:", top_user["company"]["name"])
print("Number of posts:", top_info["post_count"])
print()
print("All post titles:")
for number, title in enumerate(top_info["titles"], start=1):
    print(f"{number}. {title}")

#### What the output shows

The output shows the full name, email, company name and the numbered post titles of the user with the most posts. In the usual JSONPlaceholder data every user has exactly 10 posts, so all users are tied. In that case max() returns the first user it finds, which is Leanne Graham, and her 10 titles are printed with the numbers 1 to 10.

If the data had a clear winner, the same code would pick that user without any change.

### Part d: Filter users with two conditions

Here I am looping through all users and checking two conditions. The first one is whether the word synergies appears in the company bs field, ignoring capital letters. The second one is whether the city name has more than 8 characters. If either condition is true, the user matches. For each matching user I print the name, the city and the post count from my joined dictionary.

In [ ]:
print(f"{'Name'.ljust(26)}{'City'.ljust(18)}Posts")

matching_users = 0
for user in users:
    has_synergies = re.search(r'\bsynergies\b', user["company"]["bs"], re.IGNORECASE) is not None
    long_city = len(user["address"]["city"]) > 8

    if has_synergies or long_city:
        matching_users += 1
        print(f"{user['name'].ljust(26)}{user['address']['city'].ljust(18)}{joined[user['name']]['post_count']}")

print()
print("Total matching users:", matching_users)

#### What the output shows

The output is a small table with the name, city and post count of every user who matches at least one of my two conditions, and the last line shows how many users matched. A user is listed if the word synergies is in the company bs field or if the city name is longer than 8 characters. Because of the word boundary in my pattern, only the full word synergies is accepted. A user that meets both conditions is still printed only once, because I use a single if statement with or.

## Task 8: To Do Completion Rate Analysis Across Users

In this task I am fetching a list of to do items, checking that the data is valid, grouping the items by user, and working out how much of their work each user has finished. I am solving it in four parts.

### Part a: Fetch and validate

Here I am fetching the todos and checking three things before I trust the data. The status code must be 200, the parsed result must be a list, and the list must have at least one record. If any check fails, I print a clear error message and stop the program with sys.exit(). If there is no internet at all, the request itself fails, so I also catch that error and print a message. When all checks pass I print the total number of records.

In [ ]:
todos_url = "https://jsonplaceholder.typicode.com/todos"

try:
    response = requests.get(todos_url, timeout=15)
except requests.exceptions.RequestException as error:
    print("Error: could not reach the server.", error)
    sys.exit()

if response.status_code != 200:
    print("Error: the status code was", response.status_code, "but it should be 200.")
    sys.exit()

todos = response.json()

if not isinstance(todos, list):
    print("Error: the data I received is not a list.")
    sys.exit()

if len(todos) < 1:
    print("Error: the list has no records.")
    sys.exit()

print("Total number of todo records retrieved:", len(todos))

#### What the output shows

After all three checks pass, the cell prints the total number of records. JSONPlaceholder normally returns 200 todo items. If something went wrong, such as no internet, a wrong status code, a result that is not a list, or an empty list, the cell would print an error message and stop right there instead of continuing with bad data.

### Part b: Group todos by user

Here I am going through every todo and grouping them by userId. For each user I keep three numbers, total, completed and incomplete. Every todo adds one to the total, and then adds one to either completed or incomplete depending on its completed value. After that I print the numbers for every user, and I also check that completed plus incomplete equals total. The check prints True if the numbers match and False if they do not.

In [ ]:
user_stats = {}

for todo in todos:
    stats = user_stats.setdefault(todo["userId"], {"total": 0, "completed": 0, "incomplete": 0})
    stats["total"] += 1
    if todo["completed"]:
        stats["completed"] += 1
    else:
        stats["incomplete"] += 1

for user_id in sorted(user_stats):
    stats = user_stats[user_id]
    check = stats["completed"] + stats["incomplete"] == stats["total"]
    print(f"User {user_id}: {stats}  completed + incomplete == total: {check}")

#### What the output shows

The output has one line for each of the 10 users, and it shows the total, completed and incomplete numbers of that user. In the usual data each user has 20 todos, so every total is 20. At the end of each line the check prints True, because completed plus incomplete is always equal to the total. If any line printed False, it would mean my counting has a mistake.

### Part c: Completion rates

Here I am calculating the completion rate for every user as completed divided by total, multiplied by 100, rounded to 1 decimal place. I use max() and min() on the rates dictionary to find the user with the highest and lowest rate. The overall average is the sum of all user rates divided by the number of users, also rounded to 1 decimal place.

In [ ]:
completion_rates = {}
for user_id, stats in user_stats.items():
    completion_rates[user_id] = round(stats["completed"] / stats["total"] * 100, 1)

best_user = max(completion_rates, key=completion_rates.get)
worst_user = min(completion_rates, key=completion_rates.get)
average_rate = round(sum(completion_rates.values()) / len(completion_rates), 1)

print("Highest completion rate: userId", best_user, "with", completion_rates[best_user], "%")
print("Lowest completion rate: userId", worst_user, "with", completion_rates[worst_user], "%")
print("Overall average completion rate:", average_rate, "%")

#### What the output shows

The output shows three results. The first is the user with the highest completion rate and the rate value, and the second is the user with the lowest completion rate and its value. The third is the overall average completion rate across all users.

Each rate is a percentage with 1 decimal place, for example 45.0. The highest value is always greater than or equal to the average, and the lowest value is always less than or equal to it, which is a quick way for me to check that the results make sense.

### Part d: A neat ranking table

Here I am fetching the users so I can show names instead of id numbers. I build a lookup dictionary that maps each userId to the user's name. Then I sort the user ids by completion rate from the highest to the lowest and print a table. To keep the columns lined up, I use f string alignment, where the less than sign with a number pads the text on the right to a fixed width.

In [ ]:
users_response = requests.get("https://jsonplaceholder.typicode.com/users", timeout=15)
user_lookup = {user["id"]: user["name"] for user in users_response.json()}

ranked_ids = sorted(completion_rates, key=completion_rates.get, reverse=True)

print(f"{'Rank':<6}{'Name':<26}{'Total Tasks':<14}{'Completed':<12}{'Completion Rate %'}")
print("=" * 72)

for rank, user_id in enumerate(ranked_ids, start=1):
    name = user_lookup[user_id]
    stats = user_stats[user_id]
    print(f"{rank:<6}{name:<26}{stats['total']:<14}{stats['completed']:<12}{completion_rates[user_id]}")

#### What the output shows

The output is a table with the columns Rank, Name, Total Tasks, Completed and Completion Rate %. Rank 1 is the user with the highest completion rate, and the rates go down as the rank number goes up. The columns line up neatly because each value is padded to a fixed width.

Using names in place of user id numbers makes the table much easier to read. If two users have the same rate, they keep the order in which they were found.

## Task 9: Nested JSON Flattening, DataFrame Operations, and Comment Analysis

In this task I am taking user data that has objects inside objects, making it flat, working with it in pandas, and then counting the comments on every post. I am solving it in four parts.

### Part a: Flatten each user

Here I am fetching the users. Each user has an address object that has a geo object inside it, and also a company object. That is called nested data. I write a function called flatten_user that takes one user and returns a flat dictionary with exactly ten keys, so every value sits on the same level. Then I apply the function to all users using a list comprehension and print the final list.

In [ ]:
users_data = requests.get("https://jsonplaceholder.typicode.com/users", timeout=15).json()

def flatten_user(user: dict) -> dict:
    return {
        "id": user["id"],
        "name": user["name"],
        "username": user["username"],
        "email": user["email"],
        "city": user["address"]["city"],
        "zipcode": user["address"]["zipcode"],
        "lat": user["address"]["geo"]["lat"],
        "lng": user["address"]["geo"]["lng"],
        "company_name": user["company"]["name"],
        "company_catchphrase": user["company"]["catchPhrase"]
    }

flat_users = [flatten_user(user) for user in users_data]

for flat_user in flat_users:
    print(flat_user)

#### What the output shows

The output is a list of 10 dictionaries, one per user. Every dictionary has exactly the same ten keys, which are id, name, username, email, city, zipcode, lat, lng, company_name and company_catchphrase. The values that were hidden inside address, geo and company are now directly available at the top level.

The lat and lng values are still text at this stage, because that is how the API sends them.

### Part b: DataFrame and distance from origin

Here I am creating a pandas DataFrame from the flat list and displaying it in full. The lat and lng values come as text from the API, so I first change both columns to float numbers. Then I add a new column called distance_from_origin. For every row I use math.sqrt on lat squared plus lng squared, applied row by row with apply and axis set to 1. Finally I find the row with the biggest distance using idxmax and print that user.

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

users_df = pd.DataFrame(flat_users)
display(users_df)

users_df["lat"] = users_df["lat"].astype(float)
users_df["lng"] = users_df["lng"].astype(float)

users_df["distance_from_origin"] = users_df.apply(
    lambda row: math.sqrt(row["lat"] ** 2 + row["lng"] ** 2), axis=1
)

farthest = users_df.loc[users_df["distance_from_origin"].idxmax()]
print()
print("User with the greatest distance from origin:", farthest["name"])
print("Distance:", round(farthest["distance_from_origin"], 2))

#### What the output shows

First a full table of the 10 users is displayed, with all ten columns. After that I changed lat and lng to float numbers, so they can be used in maths, and I added the distance_from_origin column to the table. The last lines print the name of the user who has the biggest distance from the origin and the value of that distance.

The distance is the length of the straight line from the point 0, 0 on the map to the user's coordinates. A bigger number means the user's coordinates are further away from that point.

### Part c: Northern and Southern hemisphere

Here I am using the lat column to split users into two groups. A latitude above zero means the Northern Hemisphere, and a latitude below zero means the Southern Hemisphere. I print the names in each group. Then I calculate the average distance_from_origin of each group with the mean() method and print which hemisphere has the higher average.

In [ ]:
northern = users_df[users_df["lat"] > 0]
southern = users_df[users_df["lat"] < 0]

print("Northern Hemisphere users:", list(northern["name"]))
print("Southern Hemisphere users:", list(southern["name"]))
print()

northern_average = northern["distance_from_origin"].mean()
southern_average = southern["distance_from_origin"].mean()

print("Northern average distance:", round(northern_average, 2))
print("Southern average distance:", round(southern_average, 2))

if northern_average > southern_average:
    print("The Northern Hemisphere has the higher average distance from origin.")
else:
    print("The Southern Hemisphere has the higher average distance from origin.")

#### What the output shows

The output first lists the names of the Northern Hemisphere users and then the names of the Southern Hemisphere users. Together they cover all 10 users, as long as no latitude is exactly zero. After that it prints the average distance of each group and one sentence that tells which hemisphere has the higher average.

This comparison uses the distance column that I created in the previous part, and the mean() method does the averaging for each group.

### Part d: Count comments for every post

Here I am fetching the posts and the comments. Each comment has a postId that tells which post it belongs to. I use collections.Counter on the postId of all comments, which gives me the number of comments for each post. Then I build a list of tuples with the post id, the first 40 characters of the title, and the comment count. I sort this list by comment count from highest to lowest and print the top 5 and the bottom 5 posts.

In [ ]:
posts_list = requests.get("https://jsonplaceholder.typicode.com/posts", timeout=15).json()
comments_list = requests.get("https://jsonplaceholder.typicode.com/comments", timeout=15).json()

comments_per_post = Counter(comment["postId"] for comment in comments_list)

post_summary = []
for post in posts_list:
    post_summary.append((post["id"], post["title"][:40], comments_per_post.get(post["id"], 0)))

post_summary.sort(key=lambda item: item[2], reverse=True)

print("Top 5 most commented posts:")
for item in post_summary[:5]:
    print(item)

print()
print("Bottom 5 least commented posts:")
for item in post_summary[-5:]:
    print(item)

#### What the output shows

The output has two lists. The first one shows the top 5 most commented posts and the second one shows the bottom 5 least commented posts. Each line is a tuple with the post id, the first 40 characters of the title and the number of comments.

In the usual JSONPlaceholder data every post has exactly 5 comments, so all posts tie. Because the sort keeps the original order for equal counts, the top 5 are the posts with the smallest ids and the bottom 5 are the posts with the largest ids. The code would still show a real ranking if the comment counts were different.